In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_glomerulus_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if not os.path.exists(INPUT_CSV):

    raise FileNotFoundError(
        f"\nInput file not found:\n{INPUT_CSV}"
    )

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "patient_id",
    "glomerulus_id",
    "unique_glomerulus_id",
    "median_thickness_nm"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "\nThe following required columns are missing:\n"
        + "\n".join(missing_columns)
    )

print()
print("FILE 05 — GLOBAL AND PATIENT-LEVEL STATISTICS")
print("=" * 70)

print(
    f"Membrane components : {len(df)}"
)

print(
    f"Patients             : "
    f"{df['patient_id'].nunique()}"
)

print(
    f"Glomeruli            : "
    f"{df['unique_glomerulus_id'].nunique()}"
)

glomerulus_values = (
    df[
        [
            "patient_id",
            "glomerulus_id",
            "unique_glomerulus_id",
            "median_thickness_nm"
        ]
    ]
    .drop_duplicates(
        subset=["unique_glomerulus_id"]
    )
    .dropna(
        subset=["median_thickness_nm"]
    )
    .copy()
)

print()
print(
    f"Glomeruli used for statistics : "
    f"{len(glomerulus_values)}"
)

values = (
    glomerulus_values[
        "median_thickness_nm"
    ]
    .to_numpy()
)

print()
print("GLOBAL GLOMERULUS STATISTICS")
print("=" * 70)

print(
    f"Number of glomeruli : {len(values)}"
)

print(
    f"Mean thickness      : "
    f"{np.mean(values):.2f} nm"
)

print(
    f"Median thickness    : "
    f"{np.median(values):.2f} nm"
)

print(
    f"Standard deviation  : "
    f"{np.std(values, ddof=1):.2f} nm"
)

print(
    f"Minimum             : "
    f"{np.min(values):.2f} nm"
)

print(
    f"Maximum             : "
    f"{np.max(values):.2f} nm"
)

global_statistics = pd.DataFrame({

    "Statistic": [
        "Number of glomeruli",
        "Mean thickness (nm)",
        "Median thickness (nm)",
        "Standard deviation (nm)",
        "Minimum thickness (nm)",
        "Maximum thickness (nm)"
    ],

    "Value": [
        len(values),
        np.mean(values),
        np.median(values),
        np.std(values, ddof=1),
        np.min(values),
        np.max(values)
    ]
})

GLOBAL_STATS_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "global_glomerulus_statistics.csv"
)

global_statistics.to_csv(
    GLOBAL_STATS_OUTPUT,
    index=False
)

print()
print(
    "Global statistics saved:"
)

print(
    GLOBAL_STATS_OUTPUT
)

fig, ax = plt.subplots(
    figsize=(11, 8)
)

ax.boxplot(
    values,
    positions=[1],
    widths=0.45,
    patch_artist=False,
    showfliers=False
)

rng = np.random.default_rng(42)

x_jitter = rng.normal(
    loc=1,
    scale=0.045,
    size=len(values)
)

ax.scatter(
    x_jitter,
    values,
    s=35,
    alpha=0.65
)

ax.set_xticks([1])

ax.set_xticklabels(
    ["All glomeruli"]
)

ax.set_ylabel(
    "GBM thickness (nm)",
    fontsize=12
)

ax.set_title(
    "Global Distribution of GBM Thickness Across Glomeruli",
    fontsize=14,
    fontweight="bold",
    pad=15
)

explanation = (
    "How to read this plot\n\n"
    "• Each point = one glomerulus\n"
    "• Box = middle 50% of glomeruli (IQR)\n"
    "• Centre line = median thickness\n"
    "• Whiskers = 1.5 × IQR\n"
    "• Points = individual glomerulus values\n"
    "• Horizontal jitter separates overlapping points"
)

ax.text(
    1.03,
    0.98,
    explanation,
    transform=ax.transAxes,
    fontsize=9.5,
    verticalalignment="top",
    horizontalalignment="left",
    linespacing=1.4,
    bbox=dict(
        boxstyle="round,pad=0.6",
        facecolor="white",
        edgecolor="gray",
        linewidth=0.8,
        alpha=0.95
    )
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

plt.subplots_adjust(
    left=0.10,
    right=0.72,
    top=0.90,
    bottom=0.12
)

GLOBAL_PLOT = os.path.join(
    OUTPUT_FOLDER,
    "global_glomerulus_boxplot.png"
)

plt.savefig(
    GLOBAL_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print()
print(
    "Global box plot saved:"
)

print(
    GLOBAL_PLOT
)

patient_summary = (
    glomerulus_values
    .groupby("patient_id")
    .agg(

        number_of_glomeruli=(
            "unique_glomerulus_id",
            "count"
        ),

        mean_thickness_nm=(
            "median_thickness_nm",
            "mean"
        ),

        median_thickness_nm=(
            "median_thickness_nm",
            "median"
        ),

        std_thickness_nm=(
            "median_thickness_nm",
            "std"
        ),

        min_thickness_nm=(
            "median_thickness_nm",
            "min"
        ),

        max_thickness_nm=(
            "median_thickness_nm",
            "max"
        )
    )
    .reset_index()
)

patient_summary[
    "std_thickness_nm"
] = patient_summary[
    "std_thickness_nm"
].fillna(0)

PATIENT_STATS_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "gbm_patient_statistics.csv"
)

patient_summary.to_csv(
    PATIENT_STATS_OUTPUT,
    index=False
)

print()
print(
    "Patient-level statistics saved:"
)

print(
    PATIENT_STATS_OUTPUT
)

print()
print("PATIENT-LEVEL STATISTICS")
print("=" * 70)

print(
    patient_summary.to_string(
        index=False
    )
)

patients = sorted(
    glomerulus_values[
        "patient_id"
    ]
    .unique()
)

patient_data = []

for patient in patients:

    patient_values = (
        glomerulus_values[
            glomerulus_values[
                "patient_id"
            ] == patient
        ][
            "median_thickness_nm"
        ]
        .dropna()
        .to_numpy()
    )

    patient_data.append(
        patient_values
    )

fig, ax = plt.subplots(
    figsize=(14, 8)
)

ax.boxplot(
    patient_data,
    positions=np.arange(
        1,
        len(patients) + 1
    ),
    widths=0.55,
    patch_artist=False,
    showfliers=False
)

rng = np.random.default_rng(42)

for i, patient_values in enumerate(
    patient_data,
    start=1
):
    jitter = rng.normal(
        loc=i,
        scale=0.055,
        size=len(patient_values)
    )

    ax.scatter(
        jitter,
        patient_values,
        s=30,
        alpha=0.65
    )

ax.set_xticks(
    np.arange(
        1,
        len(patients) + 1
    )
)

ax.set_xticklabels(
    patients,
    rotation=45,
    ha="right"
)

ax.set_xlabel(
    "Patient ID",
    fontsize=12
)

ax.set_ylabel(
    "GBM thickness (nm)",
    fontsize=12
)

ax.set_title(
    "GBM Thickness Distribution Across Patients",
    fontsize=14,
    fontweight="bold",
    pad=15
)

explanation = (
    "How to read this plot\n\n"
    "• Each box = thickness distribution within one patient\n"
    "• Each point = one glomerulus\n"
    "• Box = middle 50% of glomeruli (IQR)\n"
    "• Centre line = patient median\n"
    "• Whiskers = 1.5 × IQR\n"
    "• Points = individual glomerulus values\n"
    "• Horizontal jitter separates overlapping points"
)

ax.text(
    1.02,
    0.98,
    explanation,
    transform=ax.transAxes,
    fontsize=9.5,
    verticalalignment="top",
    horizontalalignment="left",
    linespacing=1.4,
    bbox=dict(
        boxstyle="round,pad=0.6",
        facecolor="white",
        edgecolor="gray",
        linewidth=0.8,
        alpha=0.95
    )
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle="--"
)

plt.subplots_adjust(
    left=0.08,
    right=0.73,
    top=0.90,
    bottom=0.20
)

PATIENT_PLOT = os.path.join(
    OUTPUT_FOLDER,
    "patient_glomerulus_boxplot.png"
)

plt.savefig(
    PATIENT_PLOT,
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

print()
print(
    "Patient box plot saved:"
)

print(
    PATIENT_PLOT
)

print()
print("FILE 05 COMPLETED SUCCESSFULLY")
print("=" * 70)

print()
print("Created files:")

print(
    "1.",
    GLOBAL_STATS_OUTPUT
)

print(
    "2.",
    GLOBAL_PLOT
)

print(
    "3.",
    PATIENT_STATS_OUTPUT
)

print(
    "4.",
    PATIENT_PLOT
)

print()
print("All statistics are based on glomerulus-level values.")


FILE 05 — GLOBAL AND PATIENT-LEVEL STATISTICS
Membrane components : 257
Patients             : 11
Glomeruli            : 257

Glomeruli used for statistics : 257

GLOBAL GLOMERULUS STATISTICS
Number of glomeruli : 257
Mean thickness      : 229.74 nm
Median thickness    : 190.68 nm
Standard deviation  : 143.66 nm
Minimum             : 82.37 nm
Maximum             : 1342.94 nm

Global statistics saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\global_glomerulus_statistics.csv

Global box plot saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\global_glomerulus_boxplot.png

Patient-level statistics saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_patient_statistics.csv

PATIENT-LEVEL STATISTICS
patient_id  number_of_glomeruli  mean_thickness_nm  median_thickness_nm  std_thickness_nm  min_thickness_nm  max_thickness_nm
     01-24                   15         204.697734           174.481620         98.024550        113.405849        480.615405
  